# OLS Regression Models: Predicting Graduate Earnings
This notebook runs two OLS specifications predicting median 10-year earnings:
1. **Level-Level OLS** with state fixed effects (baseline)
2. **Log-Log OLS** with state fixed effects (preferred; elasticity interpretation)

Reads from: `data/college_scorecard_clean.csv`  
Writes to: `output/tables/` and `output/figures/`

### Setup
`stargazer` is not on conda-forge; install via pip if not already present.

In [ ]:
# Only needs to run once; safe to skip if already installed
pip install stargazer

---
## Model 1: Level-Level OLS with State Fixed Effects

**Outcome:** Median earnings 10 years after enrollment (`md_earn_10yr`)  
**Predictors:** Admission rate, SAT composite, tuition, Pell grant share, institution type  
**Fixed Effects:** State dummies absorb state-level labor market differences (e.g. California vs. Mississippi wage levels), so we compare schools *within* the same state

**Interpretation:** Coefficients are in dollar units — a 1-unit increase in X is associated with a $β change in median earnings.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer

input_path  = "../data/college_scorecard_clean.csv"
output_path = "../output/tables/ols_results.txt"

# Load & subset 
df = pd.read_csv(input_path)

Y = "md_earn_10yr"
X = ["adm_rate", "sat_composite", "tuition", "control", "pct_pell"]

# Keep only the columns we need, then drop rows with any missing values.
# Listwise deletion is conservative but keeps the estimating sample consistent
# across specifications so results are comparable.
df = df[[Y] + X + ["state"]].dropna()

print(f"Observations in estimation sample: {len(df)}")

# Estimate OLS 
# C(control) creates dummy variables for institution type; base category = private_fp
# C(state) creates 50 state dummies as fixed effects
formula = (
    "md_earn_10yr ~ adm_rate + sat_composite + tuition "
    "+ C(control) + pct_pell + C(state)"
)

model  = smf.ols(formula=formula, data=df)
result = model.fit(cov_type="HC3")  # HC3: heteroskedasticity-robust SEs (safer with ~1,500 obs)

# Print results 
# Filter out the 50 state FE rows --> they're controlled for but not the focus
main_vars = [v for v in result.params.index if not v.startswith("C(state)")]

print("\n" + "="*65)
print("OLS: Median 10-Year Earnings ~ Selectivity, SAT, Tuition, Pell, Type")
print("State Fixed Effects included  |  Robust (HC3) SEs")
print("="*65)
print(f"{'Variable':<30} {'Coef':>10} {'SE':>10} {'t':>8} {'p':>8}")
print("-"*65)
for v in main_vars:
    coef  = result.params[v]
    se    = result.bse[v]
    t     = result.tvalues[v]
    p     = result.pvalues[v]
    stars = ("***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "")
    print(f"{v:<30} {coef:>10.1f} {se:>10.1f} {t:>8.2f} {p:>8.3f} {stars}")

print("-"*65)
print(f"{'N':<30} {result.nobs:>10.0f}")
print(f"{'R-squared':<30} {result.rsquared:>10.4f}")
print(f"{'Adj. R-squared':<30} {result.rsquared_adj:>10.4f}")
print("="*65)
print("* p<0.10  ** p<0.05  *** p<0.01")

# Save full statsmodels summary to file for the appendix
with open(output_path, "w") as f:
    f.write(result.summary().as_text())
print(f"\nFull results saved to {output_path}")

#### Export Regression Table (Model 1)
Uses `stargazer` to produce a publication-style LaTeX table. Covariates are reordered so institution type dummies appear first (most policy-relevant).

In [ ]:
stargazer_m1 = Stargazer([result])

# Reorder rows: institution type dummies first, then continuous predictors
stargazer_m1.covariate_order([
    'C(control)[T.private_np]',
    'C(control)[T.public]',
    'adm_rate',
    'sat_composite',
    'tuition',
    'pct_pell',
    'Intercept'
])

# Replace generated names with readable labels for the table
stargazer_m1.rename_covariates({
    'C(control)[T.private_np]': 'Private Non-Profit',
    'C(control)[T.public]':     'Public University',
    'adm_rate':                 'Admission Rate',
    'sat_composite':            'SAT Score',
    'tuition':                  'Tuition',
    'pct_pell':                 'Pell Grant %',
    'Intercept':                'Constant'
})

stargazer_m1.add_line('State Fixed Effects', ['Yes'])
stargazer_m1.title("Level-Level OLS: Institutional Predictors of 10-Year Graduate Earnings")

with open("../output/tables/ols_results_table.tex", "w") as f:
    f.write(stargazer_m1.render_latex())

stargazer_m1  # renders HTML preview in notebook

---
## Model 2: Log-Log OLS with State Fixed Effects (Preferred Specification)

**Why log transforms?**  
Taking logs of skewed continuous variables (earnings, admission rate, tuition) compresses outliers and makes relationships more linear. More importantly, it gives **elasticity** coefficients:

> A 1% increase in X is associated with a β% change in median earnings.

**Specification:**
$$\log(\text{Earnings}) = \beta_0 + \beta_1 \log(\text{Adm Rate}) + \beta_2 \text{SAT} + \beta_3 \log(\text{Tuition}) + \beta_4 \text{Control} + \beta_5 \text{Pell\%} + \gamma_s + \varepsilon$$

SAT and Pell % are kept in levels because they are already bounded/standardized-like variables where log-transforming would not improve linearity.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer

input_path  = "../data/college_scorecard_clean.csv"
output_path = "../output/tables/ols_log_results.txt"

# Load & construct log variables 
df = pd.read_csv(input_path)

df['log_earnings'] = np.log(df['md_earn_10yr'])   # outcome
df['log_adm_rate'] = np.log(df['adm_rate'])        # log-log: coef = elasticity
df['log_tuition']  = np.log(df['tuition'])         # log-log: coef = elasticity

Y = "log_earnings"
X = ["log_adm_rate", "sat_composite", "log_tuition", "control", "pct_pell"]

# Drop rows where log transformation produced -inf (from zeros) or NaN.
# replace() catches -inf before dropna() so those rows aren't silently kept
df = df[[Y] + X + ["state"]].replace([np.inf, -np.inf], np.nan).dropna()

print(f"Observations in estimation sample: {len(df)}")

# Estimate log-log OLS 
# Same structure as Model 1; C(control) and C(state) work identically
formula = (
    "log_earnings ~ log_adm_rate + sat_composite + log_tuition "
    "+ C(control) + pct_pell + C(state)"
)

model  = smf.ols(formula=formula, data=df)
result = model.fit(cov_type="HC3")  # HC3 robust SEs

# Print results 
main_vars = [v for v in result.params.index if not v.startswith("C(state)")]

print("\n" + "="*65)
print("LOG-OLS: Log 10-Year Earnings ~ Log Adm Rate, SAT, Log Tuition, etc.")
print("State Fixed Effects included  |  Robust (HC3) SEs")
print("="*65)
print(f"{'Variable':<30} {'Coef':>10} {'SE':>10} {'t':>8} {'p':>8}")
print("-"*65)
for v in main_vars:
    coef  = result.params[v]
    se    = result.bse[v]
    t     = result.tvalues[v]
    p     = result.pvalues[v]
    stars = ("***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "")
    print(f"{v:<30} {coef:>10.4f} {se:>10.4f} {t:>8.2f} {p:>8.3f} {stars}")

print("-"*65)
print(f"{'N':<30} {result.nobs:>10.0f}")
print(f"{'R-squared':<30} {result.rsquared:>10.4f}")
print(f"{'Adj. R-squared':<30} {result.rsquared_adj:>10.4f}")
print("="*65)
print("* p<0.10  ** p<0.05  *** p<0.01")

with open(output_path, "w") as f:
    f.write(result.summary().as_text())
print(f"\nFull results saved to {output_path}")

#### Export Regression Table (Model 2)

In [ ]:
stargazer_m2 = Stargazer([result])

stargazer_m2.covariate_order([
    'C(control)[T.private_np]',
    'C(control)[T.public]',
    'log_adm_rate',
    'sat_composite',
    'log_tuition',
    'pct_pell',
    'Intercept'
])

stargazer_m2.rename_covariates({
    'C(control)[T.private_np]': 'Private Non-Profit',
    'C(control)[T.public]':     'Public University',
    'log_adm_rate':             'Log Admission Rate',
    'sat_composite':            'SAT Score',
    'log_tuition':              'Log Tuition',
    'pct_pell':                 'Pell Grant %',
    'Intercept':                'Constant'
})

stargazer_m2.add_line('State Fixed Effects', ['Yes'])
stargazer_m2.title("Log-Log OLS: Institutional Predictors of 10-Year Graduate Earnings")

with open("../output/tables/log_results_table.tex", "w") as f:
    f.write(stargazer_m2.render_latex())

stargazer_m2  # renders HTML preview in notebook

---
## Visualizations

Two scatter plots with regression lines to illustrate the two key relationships from the log-log model:
1. **Pell Grant share vs. log earnings** — the SES-earnings gradient
2. **Admission rate vs. log earnings** — the selectivity premium

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# Plot 1: Pell Grant share vs. log earnings 
# Pell % proxies for the average socioeconomic background of the student body.
# A negative slope here would suggest schools serving lower-income students
# produce lower earnings outcomes --> potentially reflects selection, not school quality.
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df,
    x='pct_pell',
    y='log_earnings',
    scatter_kws={'alpha': 0.4, 'color': '#2c3e50', 's': 15},
    line_kws={'color': '#e74c3c'}
)
plt.title('Log 10-Year Earnings vs. Pell Grant Share', fontsize=14)
plt.xlabel('Share of Pell Grant Recipients', fontsize=12)
plt.ylabel('Log Median 10-Year Earnings', fontsize=12)
plt.tight_layout()
plt.savefig('../output/figures/pell_earnings_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Admission rate vs. log earnings (non-log x-axis for readability) 
# We exponentiate log_adm_rate back to a 0–100% scale so the x-axis is intuitive.
# lowess=True fits a locally-weighted smoothing line instead of a global OLS line,
# which reveals the non-linear "elite effect" at very low admission rates.
df['adm_rate_pct'] = np.exp(df['log_adm_rate']) * 100

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.regplot(
    data=df,
    x='adm_rate_pct',
    y='log_earnings',
    scatter_kws={'alpha': 0.4, 'color': '#3498db', 's': 15},
    line_kws={'color': '#2ecc71'},
    lowess=True  # non-parametric line; better captures the curve at the selective end
)
plt.title('The Return on Selectivity: Admission Rate vs. Log Earnings', fontsize=14)
plt.xlabel('Admission Rate (%)', fontsize=12)
plt.ylabel('Log Median 10-Year Earnings', fontsize=12)
plt.xlim(0, 100)  # enforce 0–100% scale; a few schools have near-zero rates and stretch the axis
plt.tight_layout()
plt.savefig('../output/figures/selectivity_earnings_plot.png', dpi=300, bbox_inches='tight')
plt.show()